# Projeto de Previsão de Faturamento - Carro & Casa

Este notebook desenvolve um pipeline de previsão de faturamento mensal para a corretora **Carro & Casa**, utilizando dados históricos comerciais.

O objetivo é analisar indicadores mensais, treinar modelos de previsão, comparar o desempenho dos modelos e gerar cenários futuros de faturamento.

## 1. Configuração do projeto

Nesta seção, importamos as bibliotecas necessárias, conectamos o Google Drive e definimos os caminhos principais do projeto.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Montagem do Google Drive no Colab.
# Se estiver executando fora do Colab, esta célula pode ser ignorada.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as error:
    print("Google Drive não foi montado automaticamente.")
    print("Mensagem:", error)

In [ ]:
# Altere este caminho conforme o local da base no seu ambiente
project_path = "."

data_path = f"{project_path}/data"
models_path = f"{project_path}/models"
outputs_path = f"{project_path}/outputs"

# Altere este caminho conforme o local da base no seu ambiente
file_path = f"{data_path}/base_exemplo.xlsx"

# Garante que as pastas de saída existam.
os.makedirs(data_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)
os.makedirs(outputs_path, exist_ok=True)

print("Data path:", data_path)
print("Models path:", models_path)
print("Outputs path:", outputs_path)
print("File path:", file_path)

## 2. Carregamento dos dados

Nesta seção, carregamos o arquivo Excel e verificamos quais abas estão disponíveis. Para o modelo de faturamento mensal, utilizaremos a aba `comercial_mensal`.

In [ ]:
excel_file = pd.ExcelFile(file_path)

sheet_names = excel_file.sheet_names
valid_sheets = [sheet for sheet in sheet_names if sheet != "LEIA_ME"]

valid_sheets

In [ ]:
monthly_commercial = pd.read_excel(
    file_path,
    sheet_name="comercial_mensal"
)

monthly_commercial.head()

## 3. Análise inicial da base

Aqui verificamos a estrutura da base: quantidade de linhas, colunas, tipos de dados, valores ausentes e estatísticas descritivas.

In [ ]:
monthly_commercial.info()

In [ ]:
monthly_commercial.isnull().sum()

In [ ]:
monthly_commercial.describe()

In [ ]:
monthly_commercial.columns

## 4. Tratamento dos dados

Nesta etapa, convertemos a coluna de referência mensal para data, ordenamos a base cronologicamente e criamos uma base padronizada em inglês para o modelo.

In [ ]:
monthly_commercial["mes_referencia"] = pd.to_datetime(
    monthly_commercial["mes_referencia"]
)

monthly_commercial = monthly_commercial.sort_values("mes_referencia")

monthly_commercial[["mes_referencia", "faturamento_mensal"]].head()

In [ ]:
revenue_model_data = monthly_commercial[[
    "mes_referencia",
    "quantidade_vendas_mes",
    "quantidade_renovacoes_mes",
    "quantidade_cancelamentos_mes",
    "quantidade_novos_clientes_mes",
    "ticket_medio_apolice_mes",
    "taxa_retencao_estimada_percentual",
    "faturamento_mensal"
]].copy()

revenue_model_data = revenue_model_data.rename(columns={
    "mes_referencia": "month",
    "quantidade_vendas_mes": "monthly_sales",
    "quantidade_renovacoes_mes": "monthly_renewals",
    "quantidade_cancelamentos_mes": "monthly_cancellations",
    "quantidade_novos_clientes_mes": "monthly_new_customers",
    "ticket_medio_apolice_mes": "average_policy_ticket",
    "taxa_retencao_estimada_percentual": "estimated_retention_rate",
    "faturamento_mensal": "monthly_revenue"
})

revenue_model_data["month_number"] = range(1, len(revenue_model_data) + 1)
revenue_model_data["month_of_year"] = revenue_model_data["month"].dt.month

revenue_model_data.head()

## 5. Análise exploratória

Antes de treinar modelos, visualizamos o comportamento histórico do faturamento mensal.

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    revenue_model_data["month"],
    revenue_model_data["monthly_revenue"],
    marker="o"
)

plt.title("Monthly Revenue Over Time")
plt.xlabel("Month")
plt.ylabel("Monthly Revenue")
plt.grid(True)
plt.show()

## 6. Modelo baseline

O modelo baseline usa apenas a passagem do tempo (`month_number`) para prever o faturamento mensal. Ele serve como referência simples para comparação com modelos mais completos.

In [ ]:
baseline_data = revenue_model_data[["month", "month_number", "monthly_revenue"]].copy()

X_baseline = baseline_data[["month_number"]]
y_baseline = baseline_data["monthly_revenue"]

baseline_model = LinearRegression()
baseline_model.fit(X_baseline, y_baseline)

baseline_score = baseline_model.score(X_baseline, y_baseline)
baseline_score

In [ ]:
baseline_data["predicted_revenue"] = baseline_model.predict(X_baseline)

plt.figure(figsize=(12, 5))

plt.plot(
    baseline_data["month"],
    baseline_data["monthly_revenue"],
    label="Actual Revenue"
)

plt.plot(
    baseline_data["month"],
    baseline_data["predicted_revenue"],
    label="Baseline Trend"
)

plt.title("Baseline Model - Actual vs Predicted Revenue")
plt.xlabel("Month")
plt.ylabel("Monthly Revenue")
plt.legend()
plt.grid(True)
plt.show()

## 7. Modelo com variáveis comerciais

Neste modelo, utilizamos indicadores comerciais mensais para prever o faturamento. A lógica é aproximar o modelo do funcionamento real da corretora.

Variáveis utilizadas:

- número sequencial do mês;
- quantidade de vendas;
- quantidade de renovações;
- quantidade de cancelamentos;
- quantidade de novos clientes;
- ticket médio da apólice;
- taxa estimada de retenção.

In [ ]:
commercial_features = [
    "month_number",
    "monthly_sales",
    "monthly_renewals",
    "monthly_cancellations",
    "monthly_new_customers",
    "average_policy_ticket",
    "estimated_retention_rate"
]

target = "monthly_revenue"

X_commercial = revenue_model_data[commercial_features]
y_commercial = revenue_model_data[target]

commercial_model_full_fit = LinearRegression()
commercial_model_full_fit.fit(X_commercial, y_commercial)

commercial_model_full_fit.score(X_commercial, y_commercial)

## 8. Separação entre treino e teste

Como os dados são mensais, a separação é feita respeitando a ordem do tempo: os meses antigos são usados para treino e os últimos 6 meses são usados para teste.

In [ ]:
train_data = revenue_model_data.iloc[:-6]
test_data = revenue_model_data.iloc[-6:]

X_train = train_data[commercial_features]
y_train = train_data[target]

X_test = test_data[commercial_features]
y_test = test_data[target]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## 9. Avaliação da Regressão Linear

Treinamos a regressão linear usando apenas os dados de treino e avaliamos seu desempenho nos últimos 6 meses.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

linear_results = test_data[["month", "monthly_revenue"]].copy()
linear_results["predicted_revenue"] = linear_predictions
linear_results["absolute_error"] = abs(
    linear_results["monthly_revenue"] - linear_results["predicted_revenue"]
)
linear_results["percentage_error"] = (
    linear_results["absolute_error"] / linear_results["monthly_revenue"]
) * 100

linear_results

In [ ]:
linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_rmse = mean_squared_error(y_test, linear_predictions) ** 0.5
linear_r2 = r2_score(y_test, linear_predictions)

print("Linear Regression MAE:", linear_mae)
print("Linear Regression RMSE:", linear_rmse)
print("Linear Regression R²:", linear_r2)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    linear_results["month"],
    linear_results["monthly_revenue"],
    marker="o",
    label="Actual Revenue"
)

plt.plot(
    linear_results["month"],
    linear_results["predicted_revenue"],
    marker="o",
    label="Predicted Revenue"
)

plt.title("Actual vs Predicted Revenue - Test Period")
plt.xlabel("Month")
plt.ylabel("Monthly Revenue")
plt.legend()
plt.grid(True)
plt.show()

## 10. Comparação com outros modelos

Nesta seção, comparamos a regressão linear com Random Forest e uma regressão linear com uma variável simples de sazonalidade (`month_of_year`).

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = mean_squared_error(y_test, rf_predictions) ** 0.5
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest MAE:", rf_mae)
print("Random Forest RMSE:", rf_rmse)
print("Random Forest R²:", rf_r2)

In [ ]:
seasonal_features = [
    "month_number",
    "month_of_year",
    "monthly_sales",
    "monthly_renewals",
    "monthly_cancellations",
    "monthly_new_customers",
    "average_policy_ticket",
    "estimated_retention_rate"
]

X_train_seasonal = train_data[seasonal_features]
X_test_seasonal = test_data[seasonal_features]

seasonal_linear_model = LinearRegression()
seasonal_linear_model.fit(X_train_seasonal, y_train)

seasonal_predictions = seasonal_linear_model.predict(X_test_seasonal)

seasonal_mae = mean_absolute_error(y_test, seasonal_predictions)
seasonal_rmse = mean_squared_error(y_test, seasonal_predictions) ** 0.5
seasonal_r2 = r2_score(y_test, seasonal_predictions)

print("Seasonal Linear Regression MAE:", seasonal_mae)
print("Seasonal Linear Regression RMSE:", seasonal_rmse)
print("Seasonal Linear Regression R²:", seasonal_r2)

In [ ]:
model_comparison = pd.DataFrame({
    "model": [
        "Linear Regression",
        "Random Forest",
        "Seasonal Linear Regression"
    ],
    "MAE": [
        linear_mae,
        rf_mae,
        seasonal_mae
    ],
    "RMSE": [
        linear_rmse,
        rf_rmse,
        seasonal_rmse
    ],
    "R2": [
        linear_r2,
        rf_r2,
        seasonal_r2
    ]
})

model_comparison = model_comparison.sort_values("MAE").reset_index(drop=True)
model_comparison["rank"] = range(1, len(model_comparison) + 1)

model_comparison

## 11. Treinamento do modelo final

Após comparar os modelos, a regressão linear com variáveis comerciais foi escolhida como melhor modelo inicial por apresentar menor erro médio no conjunto de teste.

Para gerar previsões finais, treinamos novamente esse modelo usando todos os meses disponíveis.

In [ ]:
final_features = commercial_features.copy()

X_final = revenue_model_data[final_features]
y_final = revenue_model_data[target]

final_revenue_model = LinearRegression()
final_revenue_model.fit(X_final, y_final)

final_revenue_model.score(X_final, y_final)

In [ ]:
model_path = f"{models_path}/final_revenue_model.pkl"
features_path = f"{models_path}/final_revenue_features.pkl"

joblib.dump(final_revenue_model, model_path)
joblib.dump(final_features, features_path)

print("Modelo salvo em:", model_path)
print("Features salvas em:", features_path)

## 12. Previsão futura por cenários

Como o modelo utiliza variáveis comerciais futuras, criamos três cenários com base na média dos últimos 6 meses:

- **Conservative**: queda em vendas, renovações, novos clientes, ticket médio e retenção, com aumento nos cancelamentos;
- **Expected**: manutenção da média recente;
- **Optimistic**: melhora em vendas, renovações, novos clientes, ticket médio e retenção, com redução nos cancelamentos.

In [ ]:
recent_data = revenue_model_data.tail(6)

recent_means = recent_data[[
    "monthly_sales",
    "monthly_renewals",
    "monthly_cancellations",
    "monthly_new_customers",
    "average_policy_ticket",
    "estimated_retention_rate"
]].mean()

recent_means

In [ ]:
last_month = revenue_model_data["month"].max()
last_month_number = revenue_model_data["month_number"].max()

future_expected = pd.DataFrame({
    "month": pd.date_range(
        start=last_month + pd.DateOffset(months=1),
        periods=6,
        freq="MS"
    ),
    "month_number": range(last_month_number + 1, last_month_number + 7),
    "monthly_sales": recent_means["monthly_sales"],
    "monthly_renewals": recent_means["monthly_renewals"],
    "monthly_cancellations": recent_means["monthly_cancellations"],
    "monthly_new_customers": recent_means["monthly_new_customers"],
    "average_policy_ticket": recent_means["average_policy_ticket"],
    "estimated_retention_rate": recent_means["estimated_retention_rate"]
})

future_expected["scenario"] = "Expected"

future_expected["predicted_revenue"] = final_revenue_model.predict(
    future_expected[final_features]
)

future_expected

In [ ]:
future_conservative = future_expected.copy()

future_conservative["monthly_sales"] *= 0.90
future_conservative["monthly_renewals"] *= 0.90
future_conservative["monthly_new_customers"] *= 0.90
future_conservative["average_policy_ticket"] *= 0.95
future_conservative["estimated_retention_rate"] *= 0.97
future_conservative["monthly_cancellations"] *= 1.10

future_conservative["scenario"] = "Conservative"
future_conservative["predicted_revenue"] = final_revenue_model.predict(
    future_conservative[final_features]
)

future_optimistic = future_expected.copy()

future_optimistic["monthly_sales"] *= 1.10
future_optimistic["monthly_renewals"] *= 1.10
future_optimistic["monthly_new_customers"] *= 1.10
future_optimistic["average_policy_ticket"] *= 1.05
future_optimistic["estimated_retention_rate"] *= 1.03
future_optimistic["monthly_cancellations"] *= 0.90

future_optimistic["scenario"] = "Optimistic"
future_optimistic["predicted_revenue"] = final_revenue_model.predict(
    future_optimistic[final_features]
)

future_scenarios = pd.concat([
    future_conservative,
    future_expected,
    future_optimistic
]).reset_index(drop=True)

future_scenarios

In [ ]:
scenario_summary = future_scenarios[[
    "month",
    "scenario",
    "predicted_revenue"
]].copy()

scenario_pivot = scenario_summary.pivot(
    index="month",
    columns="scenario",
    values="predicted_revenue"
).round(2)

scenario_pivot

In [ ]:
plt.figure(figsize=(12, 5))

for scenario in future_scenarios["scenario"].unique():
    scenario_data = future_scenarios[
        future_scenarios["scenario"] == scenario
    ]

    plt.plot(
        scenario_data["month"],
        scenario_data["predicted_revenue"],
        marker="o",
        label=scenario
    )

plt.title("Future Revenue Forecast by Scenario")
plt.xlabel("Month")
plt.ylabel("Predicted Revenue")
plt.legend()
plt.grid(True)
plt.show()

## 13. Exportação dos resultados

Nesta seção, exportamos os principais resultados do projeto: comparação dos modelos, previsões futuras em formato largo e longo, resultados do teste e gráfico final.

In [ ]:
model_comparison_path = f"{outputs_path}/model_comparison.csv"
linear_results_path = f"{outputs_path}/linear_regression_test_results.csv"
future_scenarios_path = f"{outputs_path}/future_revenue_scenarios.csv"
future_scenarios_long_path = f"{outputs_path}/future_revenue_scenarios_long.csv"
chart_path = f"{outputs_path}/future_revenue_scenarios_chart.png"

model_comparison.to_csv(model_comparison_path, index=False)
linear_results.to_csv(linear_results_path, index=False)
scenario_pivot.to_csv(future_scenarios_path)
future_scenarios.to_csv(future_scenarios_long_path, index=False)

plt.figure(figsize=(12, 5))

for scenario in future_scenarios["scenario"].unique():
    scenario_data = future_scenarios[
        future_scenarios["scenario"] == scenario
    ]

    plt.plot(
        scenario_data["month"],
        scenario_data["predicted_revenue"],
        marker="o",
        label=scenario
    )

plt.title("Future Revenue Forecast by Scenario")
plt.xlabel("Month")
plt.ylabel("Predicted Revenue")
plt.legend()
plt.grid(True)
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Arquivos exportados:")
print(model_comparison_path)
print(linear_results_path)
print(future_scenarios_path)
print(future_scenarios_long_path)
print(chart_path)

## 14. Conclusão

O projeto construiu um pipeline inicial para previsão de faturamento mensal da corretora Carro & Casa.

Foram testados três modelos:

- Regressão Linear com variáveis comerciais;
- Random Forest;
- Regressão Linear com variável simples de sazonalidade.

A Regressão Linear com variáveis comerciais apresentou o menor erro médio no período de teste, sendo escolhida como melhor modelo inicial. Em seguida, o modelo final foi treinado com todos os dados disponíveis e utilizado para gerar três cenários futuros de faturamento: conservador, esperado e otimista.

Limitações importantes:

- a base possui poucos meses históricos para modelos mais complexos;
- a previsão futura depende de premissas comerciais;
- os cenários devem ser interpretados como apoio à decisão, não como previsão exata.

Próximos passos recomendados:

- ampliar a validação automática da planilha enviada pelo cliente, incluindo verificação de tipos de dados, meses duplicados e valores inconsistentes;
- atualizar mensalmente a base e reavaliar as métricas do modelo;
- testar modelos adicionais conforme o volume histórico aumentar;